# 01 · Dati — proteine umane del cervello (senza sangue), etichette multi-fonte, AlphaFold e `card_protein`

**Obiettivo del progetto.** Per ogni proteina umana espressa nel cervello, predire *quanto* si trova in ciascun compartimento della cellula
(organelli) e *dove* nel neurone (assone, dendrite, sinapsi, mielina), con un modello che usa sequenza, frammenti N/centro/C-terminale,
struttura 3D AlphaFold e un profilo testuale esperto (`card_protein`).

## Cosa cambia rispetto a Deeploc?
| Prima | Ora | Perché |
|---|---|---|
| DeepLoc con tutte le specie (~3/4 non umane) | **solo proteine umane** (DeepLoc filtrato per sequenza identica a UniProt umano) | il dominio d'uso è il proteoma umano; le specie diverse portano etichette con altre regole di curatela |
| tutte le proteine umane | **coorte cerebrale**: espresse nel cervello (HPA tessuto/cellula o UniProt) | il tool serve ai ricercatori delle malattie neuronali |
| sangue marcato ma tenuto nel training | **sangue escluso dalla coorte** | albumina, coagulazione, emoglobine non vengono localizzate dalle cellule nervose |
| fonti: DeepLoc, UniProt, HPA, OpenCell, LOPIT, BioID, MitoCarta, SynGO | **+ Gene Ontology sperimentale (GOA)** per assone/dendrite/sinapsi/mielina | senza questa fonte i compartimenti neuronali avrebbero poche decine di esempi |
| nessun testo | **`card_protein`** (6 sezioni, versione ripulita dal leakage + versione completa) | il testo porta funzione, dominio, malattia, espressione: informazione che la sequenza da sola non dà |
| feature 3D assenti | **AlphaFold**: pLDDT, disordine, esposizione dei terminali, eliche TM, toppa di lisine | un motivo C-terminale funziona solo se esposto; la mannosio-6-fosfato è un segnale *conformazionale* |
| varianti ClinVar | rimandate (vedi roadmap nel notebook 02) | non richieste in questo giro; la card già contiene le malattie |

## Le classi da predire (e perché sono così)
**Asse A — 9 organelli (partizione: le percentuali sommano a 100 %).** Stesse 9 classi di DeepLoc, così l'ancora di qualità resta confrontabile con lo stato dell'arte.

**Asse B — 5 domini neuronali (non esclusivi: probabilità di presenza).** Un mitocondrio può stare in un assone, quindi *non* si sommano ad A né tra loro.

| Classe | Fonti che la etichettano | Come entra l'etichetta | Perché serve nel cervello |
|---|---|---|---|
| Cytoplasm | DeepLoc, UniProt, HPA (citosol, citoscheletro…), OpenCell, LOPIT, BioID, GOA | include citoscheletro e grandi complessi (come DeepLoc) | tau, MAP, tubulina |
| Nucleus | idem + HPA nucleoli/speckle | | TDP-43, FUS (SLA) |
| Extracellular | DeepLoc, UniProt, GOA (senza esosomi) | solo come positivo/zero dalle fonti che la misurano: l'HPA non la vede | neuropeptidi, APOE, amiloide |
| Cell membrane | tutte | | recettori, canali |
| Mitochondrion | DeepLoc, UniProt, HPA, **MitoCarta** (inventario dell'intero genoma), LOPIT, BioID | MitoCarta dà zeri affidabili | Parkinson (PINK1), Leigh |
| Endoplasmic reticulum / Golgi | DeepLoc, UniProt, HPA, OpenCell, LOPIT | | secrezione di neurotrofine |
| Lysosome/Vacuole | DeepLoc, UniProt, HPA, GOA (**endosomi inclusi**, interruttore `ENDOSOME_IN_LYSOSOME`) | | ceroidolipofuscinosi, GBA |
| Peroxisome | DeepLoc, UniProt, HPA, LOPIT | classe rara: intervalli di confidenza ampi | adrenoleucodistrofia |
| Assone | UniProt, GOA, fonti esterne (proteomica assonale iPSC) | **solo per proteine espresse nei neuroni** | trasporto assonale, ALS |
| Dendrite | UniProt, GOA, fonti esterne | idem | spine, MAP2 |
| Presinapsi / Postsinapsi | **SynGO**, UniProt, GOA | idem | rilascio sinaptico, PSD-95 |
| Mielina | UniProt, GOA | **solo per proteine degli oligodendrociti** | PLP1, MBP |

**Regola delle assenze.** Una fonte dice 1 (presente), 0 (assente) o niente. Lo 0 vale solo per le classi che *quella fonte sa misurare* e per le proteine che *quella fonte riporta*
(HPA non misura l'extracellulare; SynGO non dice nulla di un gene che non cita). Le fonti **solo positive** (GOA) non producono mai zeri.
Il notebook 02 impara sensibilità e specificità di ogni fonte e usa anche quelle rumorose senza farsi ingannare.

## Fonti scaricate in automatico
UniProt umano (reviewed) · DeepLoc 2.0 (training + test HPA indipendente) · HPA subcellulare, RNA tessuto, RNA single-cell, proteinatlas · OpenCell ·
Hein 2024 / Itzhak 2016 (frazionamento) · LOPIT ×3 e BioID (Hein et al.) · **GOA umano + go-basic.obo** · **AlphaFold DB** (API, per proteina).

## Fonti manuali (facoltative, tranne MitoCarta consigliata)
- `cache/mitocarta3.xlsx` — broadinstitute.org/mitocarta → *Human MitoCarta3.0*.
- `data/esterni/syngo_ontologies.xlsx` — syngoportal.org → *bulk download*.
- `data/esterni/fonti/<nome>.csv` con colonne `gene,classe` (classe ∈ `Assone`, `Dendrite`, `Presinapsi`, `Postsinapsi`, `Mielina`) per le mappe neuronali. Dalla ricerca bibliografica fatta per questo progetto
  le più utili sono i supplementi di: proteomica **assonale vs somatodendritica di neuroni corticali umani iPSC** (separazione microfluidica + DIA-MS, ~6.000 proteine assonali
  e ~9.000 somatodendritiche — è un preprint, controllare la versione pubblicata), la proteomica assonale di **neuroni dopaminergici iPSC umani** (Mol Cell Proteomics 2024: 105 marcatori assonali nuovi, traduzione locale)
  e la prossimità APEX2 in neuroni dopaminergici di topo (specie diversa → fonte debole). Tutte sono tabelle supplementari: vanno scaricate a mano e convertite in due colonne.

## Output
`data/proteine.parquet` (coorte + feature + card) · `data/osservazioni.parquet` (tabella lunga proteina × fonte × classe) · `data/config_dati.json`.
Tutti i conteggi (proteine per passo del filtro, positivi per classe) vengono **stampati dal notebook** quando lo esegui: non sono riportati qui perché dipendono dalle release scaricate.

In [1]:
import gzip, io, json, re, shutil, time, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from scipy.spatial import cKDTree
from sklearn.model_selection import cross_val_predict
from sklearn.svm import SVC
from urllib3.util.retry import Retry

CACHE, DATA, EXT, AFDIR = Path('cache'), Path('data'), Path('data/esterni'), Path('cache/alphafold')
for d in [CACHE, DATA, AFDIR, EXT / 'fonti']:
    d.mkdir(parents=True, exist_ok=True)

# ---- le classi da predire -------------------------------------------------------------------------------------------
# ASSE A · ORGANELLI (9): partizione della cellula, le percentuali sommano a 100 %.  Stesse classi di DeepLoc.
# ASSE B · DOMINI NEURONALI (5): non esclusivi (un mitocondrio può stare nell'assone), restano probabilità di presenza.
ORGANELLES = ['Cytoplasm', 'Nucleus', 'Extracellular', 'Cell membrane', 'Mitochondrion',
              'Endoplasmic reticulum', 'Lysosome/Vacuole', 'Golgi apparatus', 'Peroxisome']
NEURO = ['Assone', 'Dendrite', 'Presinapsi', 'Postsinapsi', 'Mielina']     # la mielina è degli oligodendrociti
ALL = ORGANELLES + NEURO

# ---- ambito: SOLO proteine umane espresse nel cervello, SENZA proteine del sangue ------------------------------------
BRAIN_NTPM, NEURON_NCPM = 10, 5          # soglie HPA: espressione in regioni cerebrali (nTPM) e in tipi cellulari (nCPM)
ESCLUDI_SANGUE = True                    # le proteine del sangue escono dalla coorte (non solo dalle valutazioni)
ENDOSOME_IN_LYSOSOME = True               # True: la classe "Lysosome/Vacuole" include gli endosomi (Rab5/7/11, trafficking neuronale)
MAX_LEN = 12000                          # oltre questa lunghezza (titina...) la proteina esce: costo/benefici sfavorevoli
USE_ALPHAFOLD = True                     # scarica le strutture AlphaFold per la coorte (qualche ora la prima volta, poi cache)
AF_WORKERS, AF_LIMIT = 8, None           # AF_LIMIT = 200 per una prova veloce

session = requests.Session()
session.headers['User-Agent'] = 'neuroloc-brain/1.0 (academic research)'
session.mount('https://', HTTPAdapter(max_retries=Retry(total=6, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])))

In [2]:
UNIPROT_FIELDS = ('accession,gene_names,protein_name,sequence,length,cc_subcellular_location,xref_geneid,ft_signal,ft_transit,'
                  'ft_transmem,ft_lipid,keyword,cc_ptm,cc_function,cc_disease,cc_subunit,ft_region,ft_motif,xref_pdb,'
                  'cc_tissue_specificity,cc_catalytic_activity,cc_pathway,cc_activity_regulation,protein_families,'
                  'ft_domain,ft_repeat,ft_zn_fing,ft_coiled,ec')
BRAIN_QUERY = ('reviewed:true AND organism_id:9606 AND (cc_tissue_specificity:brain OR cc_tissue_specificity:neuron '
               'OR cc_tissue_specificity:neuro* OR cc_tissue_specificity:cerebr* OR cc_tissue_specificity:hippocampus '
               'OR cc_tissue_specificity:cortex OR cc_tissue_specificity:astrocyte OR cc_tissue_specificity:microglia '
               'OR cc_tissue_specificity:glia OR cc_tissue_specificity:"nervous system" OR keyword:KW-0165 '
               'OR keyword:KW-0523 OR keyword:KW-0524 OR go:0007268 OR go:0030182 OR go:0007411 OR go:0045202 '
               'OR go:0043005 OR go:0098794 OR go:0014069)')
STREAM = 'https://rest.uniprot.org/uniprotkb/stream'
DTU = 'https://services.healthtech.dtu.dk/services/DeepLoc-2.0/data/'
HPA = 'https://www.proteinatlas.org/download/tsv/'
ORGIP = 'https://raw.githubusercontent.com/czbiohub-sf/Organelle_IP_analyses_and_figures/main/'   # Hein et al., licenza BSD
GO = 'https://current.geneontology.org/'

sources = pd.DataFrame([
    ('deeploc_train.csv', DTU + 'Swissprot_Train_Validation_dataset.csv', None, False),
    ('deeploc_hpa_test.csv', DTU + 'hpa_testset.csv', None, False),
    ('uniprot_human_v4.tsv', STREAM, {'query': 'reviewed:true AND organism_id:9606 AND fragment:false',
                                      'fields': UNIPROT_FIELDS, 'format': 'tsv'}, False),
    ('uniprot_brain_ids.tsv', STREAM, {'query': BRAIN_QUERY, 'fields': 'accession', 'format': 'tsv'}, False),
    ('hpa_subcellular.zip', HPA + 'subcellular_location.tsv.zip', None, False),
    ('hpa_rna_tissue.zip', HPA + 'rna_tissue_consensus.tsv.zip', None, False),
    ('hpa_rna_single_cell.zip', HPA + 'rna_single_cell_type.tsv.zip', None, False),
    ('hpa_proteinatlas.zip', 'https://www.proteinatlas.org/download/proteinatlas.tsv.zip', None, False),
    ('synaptopathydb_genes.csv', 'https://www.synaptopathydb.org/api/genes', None, True),
    ('opencell_localization.csv', 'https://opencell.sf.czbiohub.org/data/datasets/opencell-localization-annotations.csv', None, False),
    ('hein2024_noc.csv', ORGIP + 'notebooks/enrichment/output/enrichment_and_volcano_tables/2023-10-21-imp5-for-figures_enrichment_table_NOC_prop.csv', None, False),
    ('itzhak2016.xlsx', ORGIP + 'data/external/elife-16950-supp9-v3.xlsx', None, False),
    ('lopit_hyper.csv', ORGIP + 'data/external/hyperLOPITU2OS2018.csv', None, False),
    ('lopit_dc.csv', ORGIP + 'data/external/lopitdcU2OS2018.csv', None, False),
    ('lopit_2024.csv', ORGIP + 'data/external/dLOPIT2024-DMSO.csv', None, False),
    ('marcatori_organelli.csv', ORGIP + 'data/external/curated_ground_truth_v9.0.csv', None, False),
    ('bioid2021_nmf.xlsx', ORGIP + 'data/external/BioID2021_NMF.xlsx', None, False),
    # NUOVO: annotazioni Gene Ontology con evidenza sperimentale (assone, dendrite, sinapsi, mielina e tutti gli organelli)
    ('goa_human.gaf.gz', GO + 'annotations/goa_human.gaf.gz', None, True),
    ('go-basic.obo', GO + 'ontology/go-basic.obo', None, True),
], columns=['file', 'url', 'params', 'optional'])
# MitoCarta 3.0: download manuale (Broad blocca lo scaricamento automatico) → cache/mitocarta3.xlsx

for f, url, params, optional in sources.itertuples(index=False):
    if (CACHE / f).exists():
        continue
    try:
        with session.get(url, params=params, stream=True, timeout=3600) as r, open(CACHE / f'{f}.part', 'wb') as out:
            r.raise_for_status()
            for chunk in r.iter_content(1 << 20):
                out.write(chunk)
        (CACHE / f'{f}.part').replace(CACHE / f)
        print('scaricato', f)
    except Exception as e:
        if not optional:
            raise
        print(f'ATTENZIONE: {f} non scaricato ({type(e).__name__}); la fonte verrà saltata')

scaricato goa_human.gaf.gz
scaricato go-basic.obo


## Proteine: UniProt umano, DeepLoc (solo umane), test HPA indipendente
Il dizionario simbolo→UniProt serve per collegare HPA, MitoCarta, OpenCell e le fonti esterne.

In [3]:
dl = pd.read_csv(CACHE / 'deeploc_train.csv', index_col=0).rename(
    columns={'ACC': 'uniprot_id', 'Sequence': 'sequence', 'Partition': 'partition'})
hpa_test = pd.read_csv(CACHE / 'deeploc_hpa_test.csv').rename(columns={'sid': 'id_test', 'fasta': 'sequence'})

raw = pd.read_csv(CACHE / 'uniprot_human_v4.tsv', sep='\t', dtype=str)
def col(pattern):
    """Colonna UniProt per espressione regolare sull'intestazione (le intestazioni cambiano tra release)."""
    hit = [c for c in raw.columns if re.search(pattern, c, re.I)]
    return raw[hit[0]] if hit else pd.Series(np.nan, index=raw.index, dtype=object)

up = pd.DataFrame({
    'uniprot_id': col(r'^Entry$'), 'gene_names': col(r'^Gene Names'), 'protein_name': col(r'^Protein names'),
    'sequence': col(r'^Sequence$'), 'scl': col(r'^Subcellular location'), 'geneid': col(r'^GeneID'),
    'sig': col(r'^Signal peptide'), 'transit': col(r'^Transit peptide'), 'tm': col(r'^Transmembrane'),
    'lipid': col(r'^Lipidation'), 'keywords': col(r'^Keywords'), 'ptm': col(r'^Post-translational'),
    'function': col(r'^Function'), 'disease': col(r'^Involvement in disease'), 'subunit': col(r'^Subunit'),
    'region': col(r'^Region'), 'motif': col(r'^Motif'), 'pdb': col(r'^PDB'), 'tissue': col(r'^Tissue specificity'),
    'catalytic': col(r'^Catalytic activity'), 'pathway': col(r'^Pathway'), 'actreg': col(r'^Activity regulation'),
    'family': col(r'^Protein families'), 'domain': col(r'^Domain'), 'repeat': col(r'^Repeat'),
    'zinc': col(r'^Zinc finger'), 'coiled': col(r'^Coiled coil'), 'ec': col(r'^EC number')})
up['sequence'] = up['sequence'].str.upper().str.replace(r'[^A-Z]', '', regex=True)
up['gene'] = up['gene_names'].fillna('').str.split().str[0].str.upper()
brain_ids = set(pd.read_csv(CACHE / 'uniprot_brain_ids.tsv', sep='\t')['Entry'])
print(f'UniProt umane reviewed (non frammenti): {len(up):,}')

# simbolo genico → UniProt: simbolo ufficiale, poi alias solo se non ambigui
tokens = up[['uniprot_id', 'gene_names']].assign(sym=up['gene_names'].fillna('').str.upper().str.split()).explode('sym')
alias_n = tokens.groupby('sym')['uniprot_id'].transform('nunique')
sym2id = pd.concat([up.set_index('gene')['uniprot_id'], tokens[alias_n == 1].set_index('sym')['uniprot_id']])
sym2id = sym2id[~sym2id.index.duplicated() & (sym2id.index != '')]
seq2id = up.drop_duplicates('sequence').set_index('sequence')['uniprot_id']

# DeepLoc: SOLO le proteine umane (sequenza identica a una proteina umana UniProt) → ancora di qualità ("oro")
dl['sequence'] = dl['sequence'].str.upper()
dl_h = dl[dl['sequence'].isin(seq2id.index)].copy()
dl_h['uniprot_id'] = dl_h['sequence'].map(seq2id)
dl_h = dl_h.drop_duplicates('uniprot_id')
print(f'DeepLoc totale {len(dl):,} → umane {len(dl_h):,} (le non umane escono: il target è il proteoma umano)')

# test indipendente HPA di DeepLoc 2.0: collegato a UniProt per sequenza identica
hpa_test['uniprot_id'] = hpa_test['sequence'].str.upper().map(seq2id)
print(f"Test HPA: {len(hpa_test):,} proteine, collegate a UniProt {hpa_test['uniprot_id'].notna().sum():,}")

UniProt umane reviewed (non frammenti): 20,430
DeepLoc totale 28,303 → umane 6,755 (le non umane escono: il target è il proteoma umano)
Test HPA: 1,717 proteine, collegate a UniProt 319


## Osservazioni: UniProt, HPA, MitoCarta, OpenCell, SynGO, fonti esterne, LOPIT/BioID
Ogni fonte diventa una tabella di osservazioni `(proteina, fonte, classe) → 1/0`. UniProt è diviso in `uniprot_forte` (evidenza sperimentale o curatore, codici ECO) e `uniprot_debole` (similarità/predizione). Rispetto alla versione precedente gli endosomi confluiscono in *Lysosome/Vacuole* (interruttore in cima).

In [4]:
# ---------- UniProt: ogni frase di localizzazione ha un codice di evidenza ECO ----------------------------------------
scl = up.set_index('uniprot_id')['scl'].fillna('')
scl = scl.str.replace(r'SUBCELLULAR LOCATION: \[Isoform (?!1\])[^\]]*\]:.*?(?=SUBCELLULAR LOCATION:|$)', '', regex=True)
scl = scl.str.replace(r'Note=.*?(?=SUBCELLULAR LOCATION:|$)', '', regex=True)
scl = scl.str.replace(r'SUBCELLULAR LOCATION:|\[Isoform 1\]:', '.', regex=True)
terms = scl.str.extractall(r'([A-Za-z][^.;{}]*?)\s*\{([^}]*)\}').rename(columns={0: 'term', 1: 'eco'}).reset_index()
term = terms['term'].str.strip().str.lower()
RULES = [('Cytoplasm', r'cytoplasm|cytosol|cytoskeleton|centrosome|centriole|microtubule|perikaryon|actin|spindle|p-body|stress granule'),
         ('Nucleus', r'\bnucleus\b|nucleolus|nucleoplasm|chromosome|centromere|kinetochore|telomere|pml body|cajal|nuclear (?:body|speckle|pore|envelope|membrane|matrix)'),
         ('Cell membrane', r'cell membrane|plasma membrane|cell surface|junction|sarcolemma|synaptic cell membrane'),
         ('Extracellular', r'^secreted|extracellular space|extracellular matrix|basement membrane'),
         ('Endoplasmic reticulum', r'endoplasmic reticulum|sarcoplasmic reticulum|microsome'),
         ('Golgi apparatus', r'golgi'), ('Mitochondrion', r'mitochondri'),
         ('Lysosome/Vacuole', r'lysosom|vacuol|melanosom|autophagosom' + (r'|endosom' if ENDOSOME_IN_LYSOSOME else '')),
         ('Peroxisome', r'peroxisom')]
terms['classe'] = None
for cls, pattern in RULES:
    terms.loc[term.str.contains(pattern), 'classe'] = cls
terms.loc[term.str.contains(r'single-pass|multi-pass|peripheral membrane|lipid-anchor|gpi-anchor|\bside$|^membrane$|'
                            r'cytoplasmic vesicle|exosome|granule membrane'), 'classe'] = None
neuro_term = pd.Series(None, index=terms.index, dtype=object)
for cls, pattern in [('Assone', r'\baxon'), ('Dendrite', r'dendrit|spine'),
                     ('Presinapsi', r'presynap|synaptic vesicle|active zone|synaptic bouton'),
                     ('Postsinapsi', r'postsynap|postsynaptic density'), ('Mielina', r'myelin')]:
    neuro_term[term.str.contains(pattern)] = cls
terms = pd.concat([terms, terms.assign(classe=neuro_term)], ignore_index=True).dropna(subset=['classe'])
strong = terms['eco'].str.contains(r'ECO:0000269|ECO:0000314|ECO:0007005|ECO:0000353|ECO:0000305')
obs_uniprot = terms.assign(fonte=np.where(strong, 'uniprot_forte', 'uniprot_debole'))[['uniprot_id', 'fonte', 'classe']]

# ---------- HPA subcellulare ------------------------------------------------------------------------------------------
with zipfile.ZipFile(CACHE / 'hpa_subcellular.zip') as z:
    hpa = pd.read_csv(z.open(z.namelist()[0]), sep='\t')
HPA_MAP = {'Nucleoplasm': 'Nucleus', 'Nuclear speckles': 'Nucleus', 'Nuclear membrane': 'Nucleus', 'Nucleoli': 'Nucleus',
           'Nucleoli fibrillar center': 'Nucleus', 'Nucleoli rim': 'Nucleus', 'Nuclear bodies': 'Nucleus', 'Kinetochore': 'Nucleus',
           'Mitotic chromosome': 'Nucleus', 'Cytosol': 'Cytoplasm', 'Centrosome': 'Cytoplasm', 'Centriolar satellite': 'Cytoplasm',
           'Microtubules': 'Cytoplasm', 'Intermediate filaments': 'Cytoplasm', 'Actin filaments': 'Cytoplasm', 'Aggresome': 'Cytoplasm',
           'Cytoplasmic bodies': 'Cytoplasm', 'Mitotic spindle': 'Cytoplasm', 'Plasma membrane': 'Cell membrane',
           'Cell Junctions': 'Cell membrane', 'Focal adhesion sites': 'Cell membrane', 'Endoplasmic reticulum': 'Endoplasmic reticulum',
           'Golgi apparatus': 'Golgi apparatus', 'Mitochondria': 'Mitochondrion', 'Lysosomes': 'Lysosome/Vacuole',
           'Peroxisomes': 'Peroxisome'}
if ENDOSOME_IN_LYSOSOME:
    HPA_MAP['Endosomes'] = 'Lysosome/Vacuole'
hpa_long = hpa.melt(id_vars='Gene name', value_vars=['Enhanced', 'Supported', 'Approved'], var_name='rel', value_name='loc').dropna()
hpa_long = hpa_long.assign(loc=hpa_long['loc'].str.split(';')).explode('loc')
hpa_long['classe'] = hpa_long['loc'].str.strip().map(HPA_MAP)
hpa_long['fonte'] = np.where(hpa_long['rel'] == 'Approved', 'hpa_approved', 'hpa_alta')
hpa_long['uniprot_id'] = hpa_long['Gene name'].str.upper().map(sym2id)
obs_hpa = hpa_long.dropna(subset=['uniprot_id'])[['uniprot_id', 'fonte', 'classe']]

# ---------- MitoCarta 3.0 (inventario dell'intero genoma: per lei ogni proteina assente è uno 0) ---------------------------
_mito_path = CACHE / 'mitocarta3.xlsx'
mito_ids = set()
if _mito_path.exists():
    mito = pd.read_excel(_mito_path, sheet_name='A Human MitoCarta3.0')
    mito_ids = set(mito['Symbol'].astype(str).str.upper().map(sym2id).dropna())
else:
    print('ATTENZIONE: manca cache/mitocarta3.xlsx (broadinstitute.org/mitocarta → Human MitoCarta3.0): la fonte viene saltata')

# ---------- OpenCell (HEK293T, tag endogeno): gradi 3 e 2 = netto; grado 1 = né presente né assente -----------------------
OC_MAP = {'cytoplasmic': 'Cytoplasm', 'cytoskeleton': 'Cytoplasm', 'centrosome': 'Cytoplasm', 'nucleoplasm': 'Nucleus',
          'nucleolus_gc': 'Nucleus', 'nucleolus_fc_dfc': 'Nucleus', 'nuclear_punctae': 'Nucleus', 'nuclear_membrane': 'Nucleus',
          'chromatin': 'Nucleus', 'membrane': 'Cell membrane', 'cell_contact': 'Cell membrane', 'focal_adhesions': 'Cell membrane',
          'er': 'Endoplasmic reticulum', 'golgi': 'Golgi apparatus', 'mitochondria': 'Mitochondrion'}
oc = pd.read_csv(CACHE / 'opencell_localization.csv')
oc = oc.melt(id_vars='target_name', value_vars=['annotations_grade_3', 'annotations_grade_2', 'annotations_grade_1'],
             var_name='grade', value_name='loc').dropna()
oc = oc.assign(loc=oc['loc'].str.split(r'[;,]')).explode('loc')
oc['classe'] = oc['loc'].str.strip().map(OC_MAP)
oc['uniprot_id'] = oc['target_name'].str.upper().map(sym2id)
oc['fonte'] = 'opencell'
obs_oc = oc[oc['grade'] != 'annotations_grade_1'].dropna(subset=['uniprot_id'])[['uniprot_id', 'fonte', 'classe']]
oc_weak = oc[oc['grade'] == 'annotations_grade_1'].dropna(subset=['uniprot_id', 'classe'])[['uniprot_id', 'fonte', 'classe']]

# ---------- SynGO (manuale: data/esterni/syngo_ontologies.xlsx) → presinapsi / postsinapsi --------------------------------
sg_files = sorted(EXT.glob('syngo_ontologies*.xlsx'))
sg = pd.concat([pd.read_excel(f) for f in sg_files] + [pd.DataFrame()], ignore_index=True)
sg.columns = sg.columns.astype(str).str.lower().str.replace(r'[^a-z0_]+', '', regex=True)
sg = sg.reindex(columns=['gotermname', 'godomain', 'geneshgnc_symbol']).astype(str)
sg['classe'] = np.select([sg['gotermname'].str.contains(r'(?i)presynap|synaptic vesicle|active zone'),
                          sg['gotermname'].str.contains(r'(?i)postsynap')], ['Presinapsi', 'Postsinapsi'], 'nessuna')
sg = sg[sg['godomain'].str.contains(r'(?i)^cc$|cellular') & (sg['classe'] != 'nessuna')]
sg = sg.assign(gene=sg['geneshgnc_symbol'].str.split(r'\s*[,;]\s*')).explode('gene')
sg['uniprot_id'] = sg['gene'].str.upper().map(sym2id)
obs_syngo = sg.dropna(subset=['uniprot_id']).assign(fonte='syngo')[['uniprot_id', 'fonte', 'classe']]

# ---------- fonti esterne neuronali (manuali): data/esterni/fonti/<nome>.csv con colonne gene, classe -----------------------
# Esempi consigliati (supplementi pubblici, converti in due colonne):
#   · proteomica assonale vs somatodendritica di neuroni umani iPSC (microfluidica + DIA-MS) → classi Assone / Dendrite
#   · proteomica assonale di neuroni dopaminergici iPSC umani (Mol Cell Proteomics 2024)
#   · prossimità APEX2 in neuroni dopaminergici di topo (assone vs somatodendrite) — specie diversa, usare come fonte debole
ext_files = sorted((EXT / 'fonti').glob('*.csv'))
ext = pd.concat([pd.read_csv(f).assign(fonte=f.stem) for f in ext_files] + [pd.DataFrame(columns=['gene', 'classe', 'fonte'])], ignore_index=True)
ext['uniprot_id'] = ext['gene'].astype(str).str.upper().map(sym2id)
obs_ext = ext.dropna(subset=['uniprot_id'])[['uniprot_id', 'fonte', 'classe']]
print('Fonti esterne:', [f.stem for f in ext_files] or 'nessuna (il notebook funziona lo stesso)')

# ---------- LOPIT e BioID (proteomica spaziale umana, classificatore sui marcatori) ---------------------------------------
MK_MAP = {'nucleus': 'Nucleus', 'nuclear_pore': 'Nucleus', 'cytosol': 'Cytoplasm', 'ribosome': 'Cytoplasm', 'stress granule': 'Cytoplasm',
          'centrosome': 'Cytoplasm', 'actin-binding_protein': 'Cytoplasm', 'p-body': 'Cytoplasm', 'mitochondria': 'Mitochondrion',
          'ER': 'Endoplasmic reticulum', 'ERGIC': 'Endoplasmic reticulum', 'plasma membrane': 'Cell membrane',
          'lysosome': 'Lysosome/Vacuole', 'Golgi': 'Golgi apparatus', 'peroxisome': 'Peroxisome'}
markers = pd.read_csv(CACHE / 'marcatori_organelli.csv')
markers = pd.DataFrame({'uniprot_id': markers['gene_name_canonical'].str.upper().map(sym2id),
                        'classe': markers['compartment'].map(MK_MAP)}).dropna().drop_duplicates('uniprot_id').set_index('uniprot_id')['classe']
obs_lopit = []
for name, f in [('lopit_hyper', 'lopit_hyper.csv'), ('lopit_dc', 'lopit_dc.csv'), ('lopit_2024', 'lopit_2024.csv')]:
    prof = pd.read_csv(CACHE / f, index_col=0).dropna()
    prof.index = prof.index.astype(str).str.split(';').str[0].str.replace(r'-\d+$', '', regex=True)
    prof = prof[~prof.index.duplicated()]
    lab = markers.reindex(prof.index)
    is_mk = lab.notna().to_numpy()
    svm = SVC(C=10, gamma='scale', probability=True, class_weight='balanced', random_state=0)
    prob = np.zeros((len(prof), lab[is_mk].nunique()))
    prob[is_mk] = cross_val_predict(svm, prof.values[is_mk], lab[is_mk], cv=5, method='predict_proba')
    prob[~is_mk] = svm.fit(prof.values[is_mk], lab[is_mk]).predict_proba(prof.values[~is_mk])
    call = pd.DataFrame({'uniprot_id': prof.index, 'classe': svm.classes_[prob.argmax(1)], 'p': prob.max(1)})
    obs_lopit.append(call[call['p'] >= 0.7].assign(fonte=name)[['uniprot_id', 'fonte', 'classe']])
    print(f"{name}: {len(prof):,} proteine, assegnate con p ≥ 0,7: {(call['p'] >= 0.7).sum():,}")

nmf = pd.read_excel(CACHE / 'bioid2021_nmf.xlsx', sheet_name='A) rank profiles').set_index('gene')
rank_term = pd.read_excel(CACHE / 'bioid2021_nmf.xlsx', sheet_name='B) enriched terms').groupby('rank').head(1).set_index('rank')['term name'].str.lower()
rank_cls = pd.Series('nessuna', index=rank_term.index)
for pattern, cls in [(r'cytosol|cytoskeleton|centrosome|ribonucleoprotein', 'Cytoplasm'), (r'\bnucle', 'Nucleus'),
                     (r'mitochondri', 'Mitochondrion'), (r'endoplasmic', 'Endoplasmic reticulum'), (r'golgi', 'Golgi apparatus'),
                     (r'plasma membrane|cell periphery', 'Cell membrane'), (r'lysosom', 'Lysosome/Vacuole'), (r'peroxisom', 'Peroxisome')]:
    rank_cls[rank_term.str.contains(pattern)] = cls
w = nmf.set_axis(nmf.columns.astype(str).str.split().str[-1].astype(int), axis=1).T.groupby(rank_cls).sum().T.drop(columns='nessuna', errors='ignore')
w = w.div(nmf.sum(axis=1).replace(0, np.nan), axis=0)
bio = w.stack().rename('quota').reset_index().rename(columns={'level_1': 'classe'})
bio = bio[bio['quota'] >= 0.35]
obs_bioid = pd.DataFrame({'uniprot_id': bio['gene'].astype(str).str.upper().map(sym2id), 'fonte': 'bioid', 'classe': bio['classe']}).dropna()

Fonti esterne: nessuna (il notebook funziona lo stesso)
lopit_hyper: 4,869 proteine, assegnate con p ≥ 0,7: 4,058
lopit_dc: 6,765 proteine, assegnate con p ≥ 0,7: 5,655
lopit_2024: 5,314 proteine, assegnate con p ≥ 0,7: 3,953


C:\Users\cesar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
C:\Users\cesar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


## Gene Ontology sperimentale (GOA) → assone, dendrite, sinapsi, mielina
Le annotazioni con evidenza sperimentale (EXP, IDA, IPI, IMP, IGI, IEP e le varianti ad alta processività) vengono propagate ai discendenti del termine GO (relazioni `is_a` e `part_of`), esclusi gli esosomi. È una fonte *solo positiva*.

In [5]:
# ---------- NUOVO · Gene Ontology (GOA umano) con evidenza SPERIMENTALE ---------------------------------------------------
# Serve soprattutto ad avere esempi di assone, dendrite, presinapsi, postsinapsi e mielina oltre a SynGO.
# Le annotazioni GO sono "a mondo aperto": se manca un'annotazione non vuol dire assenza → fonte SOLO POSITIVA (nessuno 0).
# Il GO NON entra mai come feature (conterrebbe la risposta): è usato solo per costruire etichette.
EXP = {'EXP', 'IDA', 'IPI', 'IMP', 'IGI', 'IEP', 'HTP', 'HDA', 'HMP', 'HGI', 'HEP'}
GO_ROOTS = {'Cytoplasm': ['GO:0005829', 'GO:0005856'], 'Nucleus': ['GO:0005634'], 'Extracellular': ['GO:0005576'],
            'Cell membrane': ['GO:0005886'], 'Mitochondrion': ['GO:0005739'], 'Endoplasmic reticulum': ['GO:0005783'],
            'Lysosome/Vacuole': ['GO:0005764', 'GO:0005773'] + (['GO:0005768'] if ENDOSOME_IN_LYSOSOME else []),
            'Golgi apparatus': ['GO:0005794'], 'Peroxisome': ['GO:0005777'],
            'Assone': ['GO:0030424'], 'Dendrite': ['GO:0030425'], 'Presinapsi': ['GO:0098793'],
            'Postsinapsi': ['GO:0098794'], 'Mielina': ['GO:0043209']}
GO_EXCLUDE = {'Extracellular': ['GO:0070062', 'GO:1903561', 'GO:0065010']}     # esosomi / vescicole extracellulari: rumore dei proteomi

def parse_obo(path):
    """Relazioni figlio → genitori (is_a e part_of) dal file go-basic.obo."""
    parents, cur, obsolete = {}, None, set()
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if line == '[Term]':
                cur = None
            elif line.startswith('id: GO:'):
                cur = line[4:14]
                parents.setdefault(cur, set())
            elif cur and line.startswith('is_a: GO:'):
                parents[cur].add(line[6:16])
            elif cur and line.startswith('relationship: part_of GO:'):
                parents[cur].add(line[22:32])
            elif cur and line == 'is_obsolete: true':
                obsolete.add(cur)
    return parents, obsolete

def descendants(parents, roots):
    kids = {}
    for child, ps in parents.items():
        for p in ps:
            kids.setdefault(p, set()).add(child)
    seen, stack = set(roots), list(roots)
    while stack:
        for k in kids.get(stack.pop(), ()):
            if k not in seen:
                seen.add(k)
                stack.append(k)
    return seen

def goa_observations(gaf_path, obo_path):
    parents, obsolete = parse_obo(obo_path)
    go2cls = {}
    for cls, roots in GO_ROOTS.items():
        ids = descendants(parents, roots) - obsolete
        if cls in GO_EXCLUDE:
            ids -= descendants(parents, GO_EXCLUDE[cls])
        for g in ids:
            go2cls.setdefault(g, []).append(cls)
    gaf = pd.read_csv(gaf_path, sep='\t', comment='!', header=None, usecols=[1, 3, 4, 6, 8],
                      names=['uniprot_id', 'qual', 'go', 'ev', 'aspect'], dtype=str)
    gaf = gaf[(gaf['aspect'] == 'C') & gaf['ev'].isin(EXP) & ~gaf['qual'].fillna('').str.contains('NOT|colocalizes_with')]
    gaf['classe'] = gaf['go'].map(go2cls)
    gaf = gaf.dropna(subset=['classe']).explode('classe')
    return gaf[['uniprot_id', 'classe']].drop_duplicates().assign(fonte='goa_exp', obs=1.0)

obs_goa = pd.DataFrame(columns=['uniprot_id', 'classe', 'fonte', 'obs'])
if (CACHE / 'goa_human.gaf.gz').exists() and (CACHE / 'go-basic.obo').exists():
    obs_goa = goa_observations(CACHE / 'goa_human.gaf.gz', CACHE / 'go-basic.obo')
    print('GO sperimentale: proteine per classe\n', obs_goa.groupby('classe')['uniprot_id'].nunique().reindex(ALL).to_string())

# ---------- regola delle assenze → tabella lunga di osservazioni -------------------------------------------------------------
# Per ogni fonte: proteine riportate × classi che la fonte sa misurare. Presente = 1, altrimenti 0.
pos = pd.concat([obs_uniprot, obs_hpa, obs_oc, obs_syngo, obs_ext, obs_bioid] + obs_lopit, ignore_index=True)
covered = pos[['uniprot_id', 'fonte']].drop_duplicates()
measured = pos.dropna(subset=['classe'])[['fonte', 'classe']].drop_duplicates()
grid = covered.merge(measured, on='fonte')
hits = pos.dropna(subset=['classe']).drop_duplicates().assign(obs=1.0)
obs = grid.merge(hits, on=['uniprot_id', 'fonte', 'classe'], how='left').fillna({'obs': 0.0})
obs = obs.merge(oc_weak.drop_duplicates().assign(debole=1), on=['uniprot_id', 'fonte', 'classe'], how='left')
obs = obs[obs['debole'].isna() | (obs['obs'] == 1)].drop(columns='debole')
obs_mito = pd.DataFrame({'uniprot_id': up['uniprot_id'], 'fonte': 'mitocarta', 'classe': 'Mitochondrion',
                         'obs': up['uniprot_id'].isin(mito_ids).astype(float)}) if mito_ids else pd.DataFrame(columns=obs.columns)
# DeepLoc (umano) = fonte "oro": tutte e 9 le classi di organello osservate per ogni proteina
obs_dl = dl_h.melt(id_vars='uniprot_id', value_vars=ORGANELLES, var_name='classe', value_name='obs').assign(fonte='deeploc')
obs = pd.concat([obs_dl, obs, obs_mito, obs_goa], ignore_index=True)[['uniprot_id', 'fonte', 'classe', 'obs']]
obs = obs[obs['classe'].isin(ALL)].drop_duplicates(['uniprot_id', 'fonte', 'classe'])
POSITIVE_ONLY = ['goa_exp']          # fonti senza zeri: nel notebook 02 hanno specificità fissata, non appresa
print(obs.groupby('fonte').agg(proteine=('uniprot_id', 'nunique'), osservazioni=('obs', 'size'), positivi=('obs', 'sum')).astype(int))

GO sperimentale: proteine per classe
 classe
Cytoplasm                3349
Nucleus                  5091
Extracellular            1642
Cell membrane            2868
Mitochondrion            1485
Endoplasmic reticulum    1156
Lysosome/Vacuole          959
Golgi apparatus           830
Peroxisome                100
Assone                     53
Dendrite                   67
Presinapsi                 47
Postsinapsi                90
Mielina                     4
                proteine  osservazioni  positivi
fonte                                           
bioid               3959         23754      4155
deeploc             6755         60795      9347
goa_exp            12206         17741     17741
hpa_alta            6404         51232      7140
hpa_approved        7847         62776      9725
lopit_2024          3953         27671      3953
lopit_dc            5655         45240      5655
lopit_hyper         4058         32464      4058
mitocarta          20430         20430      1

## Contesto: target ausiliari, espressione cerebrale, sangue, quote misurate
I **target ausiliari** (peptide segnale, TM, NLS, PTM…) sono meccanismi di sorting che il modello deve *predire dalla sequenza*: non sono mai input. Le **quote misurate** (citosol/nucleo da frazionamento in HEK293T e HeLa) sono le uniche percentuali reali disponibili su scala proteomica: addestrano le teste delle percentuali. Limite dichiarato: sono linee tumorali, non neuroni.

In [6]:
# ---------- target ausiliari (meccanismi di sorting): il modello li PREDICE dalla sequenza, non sono mai input ---------------
aux = pd.DataFrame({'uniprot_id': up['uniprot_id'],
                    'aux_peptide_segnale': up['sig'].notna().astype(float),
                    'aux_transit_peptide': up['transit'].notna().astype(float),
                    'aux_tm_1piu': up['tm'].fillna('').str.count('TRANSMEM').ge(1).astype(float),
                    'aux_tm_2piu': up['tm'].fillna('').str.count('TRANSMEM').ge(2).astype(float),
                    'aux_ancora_lipidica': (up['lipid'].notna() | up['keywords'].fillna('').str.contains('GPI-anchor|Lipoprotein')).astype(float),
                    'aux_glicoproteina': up['keywords'].fillna('').str.contains('Glycoprotein').astype(float)})
fkw = (up['function'].fillna('') + ' ' + up['subunit'].fillna('') + ' ' + up['keywords'].fillna('')).str.lower()
ptm = (up['ptm'].fillna('') + ' ' + up['keywords'].fillna('')).str.lower()
dis = up['disease'].fillna('').str.lower()
syn_genes = set()
if (CACHE / 'synaptopathydb_genes.csv').exists():
    syn_genes = set(pd.read_csv(CACHE / 'synaptopathydb_genes.csv')['HumanName'].astype(str).str.upper())
aux['aux_nls'] = (up['region'].fillna('') + up['motif'].fillna('') + up['keywords'].fillna('')).str.contains('(?i)nuclear localization signal')
aux['aux_multimero'] = fkw.str.contains(r'\bhomodimer|\bheterodimer|\bdimer|\btrimer|\btetramer|\boligomer|\bmultimer')
aux['aux_fosforilazione'] = ptm.str.contains(r'phosphorylat|phosphoprotein')
aux['aux_ubiquitinazione'] = ptm.str.contains(r'ubiquitinat|ubiquitylat|ubl conjugation')
aux['aux_palmitoilazione'] = ptm.str.contains(r'palmitoylat|palmitate')
aux['aux_processamento'] = ptm.str.contains(r'proteolytic|cleav|propeptide|zymogen')
aux['aux_aggregati'] = (dis + ' ' + ptm + ' ' + fkw).str.contains(r'amyloid|lewy bod|neurofibrillary tangle|inclusion bod|protein inclusion|aggregat')
aux['aux_patologia_cerebrale'] = dis.str.contains(r'alzheimer|parkinson|epilep|encephalopath|dementia|neurodegenerat|huntington|ataxia|intellectual disability|amyotrophic|leukodystroph|microcephal')
aux['aux_sinaptopatia'] = up['gene'].isin(syn_genes)
aux[aux.columns[1:]] = aux[aux.columns[1:]].astype(float)

# ---------- espressione cerebrale: tessuto (HPA RNA) e tipo cellulare (HPA single-cell) -----------------------------------
with zipfile.ZipFile(CACHE / 'hpa_rna_tissue.zip') as z:
    rna = pd.read_csv(z.open(z.namelist()[0]), sep='\t', usecols=['Gene name', 'Tissue', 'nTPM'])
rna['g'] = rna['Gene name'].str.upper()
BRAIN_TISSUES = ['cerebral cortex', 'cerebellum', 'hippocampal formation', 'hippocampus', 'basal ganglia', 'hypothalamus', 'amygdala',
                 'midbrain', 'pons and medulla', 'pons', 'medulla oblongata', 'spinal cord', 'white matter', 'thalamus', 'choroid plexus']
rb = rna[rna['Tissue'].str.lower().isin(BRAIN_TISSUES)]
brain_ntpm = rb.groupby('g')['nTPM'].max()
top_regions = (rb.sort_values('nTPM', ascending=False).groupby('g').head(3)
               .assign(t=lambda d: d['Tissue'] + ' (' + d['nTPM'].fillna(0).round(0).astype(int).astype(str) + ' nTPM)')
               .groupby('g')['t'].agg(', '.join))

with zipfile.ZipFile(CACHE / 'hpa_rna_single_cell.zip') as z:
    sc = pd.read_csv(z.open(z.namelist()[0]), sep='\t', usecols=['Gene name', 'Cell type', 'nCPM'])
sc['g'] = sc['Gene name'].str.upper()
neuron_ncpm = sc[sc['Cell type'].str.contains('neuron', case=False)].groupby('g')['nCPM'].max()
glia_ncpm = sc[sc['Cell type'].str.contains('astrocyte|oligodendro|microglia', case=False)].groupby('g')['nCPM'].max()
oligo_ncpm = sc[sc['Cell type'].str.contains('oligodendrocyte', case=False)].groupby('g')['nCPM'].max()
sb = sc[sc['Cell type'].str.contains('neuron|astrocyte|oligodendro|microglia', case=False)]
top_cells = (sb.sort_values('nCPM', ascending=False).groupby('g').head(3)
             .assign(t=lambda d: d['Cell type'] + ' (' + d['nCPM'].fillna(0).round(0).astype(int).astype(str) + ' nCPM)')
             .groupby('g')['t'].agg(', '.join))

ctx = up[['uniprot_id', 'gene', 'geneid', 'protein_name']].copy()
ctx['brain_ntpm'] = ctx['gene'].map(brain_ntpm)
ctx['neuron_ncpm'] = ctx['gene'].map(neuron_ncpm)
ctx['glia_ncpm'] = ctx['gene'].map(glia_ncpm)
ctx['brain_regions'] = ctx['gene'].map(top_regions).fillna('')
ctx['brain_celltypes'] = ctx['gene'].map(top_cells).fillna('')
ctx['is_brain'] = (ctx['uniprot_id'].isin(brain_ids) | (ctx['brain_ntpm'] >= BRAIN_NTPM)).astype(int)
ctx['is_neuronal'] = (ctx['neuron_ncpm'] >= NEURON_NCPM).astype(int)
ctx['is_oligo'] = (ctx['gene'].map(oligo_ncpm) >= NEURON_NCPM).astype(int)

# ---------- SANGUE: proteine plasmatiche (fegato), globuli rossi e leucociti non appartengono al contesto cerebrale -----------
with zipfile.ZipFile(CACHE / 'hpa_proteinatlas.zip') as z:
    atlas = pd.read_csv(z.open(z.namelist()[0]), sep='\t', low_memory=False)
pick = pd.concat([atlas.filter(regex='(?i)^gene$'), atlas.filter(regex='(?i)secretome location').iloc[:, :1],
                  atlas.filter(regex='(?i)blood cell specificity').iloc[:, :1], pd.DataFrame(index=atlas.index, columns=['_'])], axis=1)
pick.columns = ['gene_hpa', 'secretome', 'blood_cell'][:pick.shape[1] - 1] + ['_']
pick = pick.reindex(columns=['gene_hpa', 'secretome', 'blood_cell']).astype(str)
pick['gene_hpa'] = pick['gene_hpa'].str.upper()
to_blood = pick['secretome'].str.contains('(?i)blood') & ~pick['secretome'].str.contains('(?i)brain')
blood_cell = pick['blood_cell'].str.contains('(?i)enriched|enhanced')
ctx['secreto_cervello'] = ctx['gene'].isin(pick.loc[pick['secretome'].str.contains('(?i)brain'), 'gene_hpa']).astype(int)
ctx['is_blood'] = (ctx['gene'].isin(pick.loc[to_blood, 'gene_hpa']) | (ctx['gene'].isin(pick.loc[blood_cell, 'gene_hpa']) & ~(ctx['brain_ntpm'] >= BRAIN_NTPM))
                   | ctx['gene'].str.match(r'^HB[ABDEGMQZ]\d?$')).astype(int)

# ---------- contesto genico (multi-localizzazione, ruolo) -----------------------------------------------------------------
hpa_cls = hpa_long.dropna(subset=['classe', 'uniprot_id']).groupby('uniprot_id')['classe']
up_cls = obs_uniprot[obs_uniprot['classe'].isin(ORGANELLES)].groupby('uniprot_id')['classe']
ROLES = [('sorting receptor', r'sorting receptor|clathrin|endocytosis|vesicle trafficking'), ('channel', r'\bchannel\b|\bpore\b|\btransporter\b'),
         ('enzyme', r'\bkinase\b|\bphosphatase\b|\bprotease\b|\bligase\b|\bhydrolase\b|\bsynthase\b|\breductase\b'),
         ('receptor', r'\breceptor\b'), ('structural', r'\bstructural\b|\bscaffold\b|\bcytoskeletal\b|\bfilament\b')]
ctx['ctx_n_localizzazioni_hpa'] = ctx['uniprot_id'].map(hpa_cls.nunique())
ctx['ctx_shuttle_hpa'] = ctx['uniprot_id'].map(hpa_cls.agg(set).map({'Nucleus', 'Cytoplasm'}.issubset)).astype(float)
ctx['ctx_struttura_pdb'] = up['pdb'].notna().astype(float).to_numpy()
ctx['ctx_ruolo_trafficking'] = np.select([fkw.str.contains(p).to_numpy() for _, p in ROLES], np.arange(1, 6), 6)
ctx = ctx.merge(aux, on='uniprot_id', how='left')

# ---------- percentuali MISURATE: quota citosolica e nucleare (Hein 2024 HEK293T, Itzhak 2016 HeLa) -----------------------
noc = pd.read_csv(CACHE / 'hein2024_noc.csv', header=[0, 1], low_memory=False)
hein = pd.DataFrame({'ids': noc['metadata']['Majority protein IDs'].astype(str), 'frac_cyto_hek': noc['sample']['NOC_cytosol'],
                     'frac_nuc_hek': noc['sample']['NOC_nuclear']})
itz = pd.read_excel(CACHE / 'itzhak2016.xlsx', sheet_name='Intensities Static')
itz_tot = itz.filter(regex=r'Intensity MAP\d_(?i:nuclear|cytosolic|organellar)').sum(axis=1)
itz = pd.DataFrame({'ids': itz['Majority protein IDs'].astype(str),
                    'frac_cyto_hela': itz.filter(regex=r'Intensity MAP\d_cytosolic').sum(axis=1) / itz_tot.where(itz_tot > 0),
                    'frac_nuc_hela': itz.filter(regex=r'Intensity MAP\d_(?i:nuclear)').sum(axis=1) / itz_tot.where(itz_tot > 0)})
up_ids, frac = set(up['uniprot_id']), []
for tab in [hein, itz]:
    t = tab.assign(uniprot_id=tab['ids'].str.split(';')).explode('uniprot_id')
    t['uniprot_id'] = t['uniprot_id'].str.replace(r'-\d+$', '', regex=True)
    frac.append(t[t['uniprot_id'].isin(up_ids)].drop(columns='ids').drop_duplicates('uniprot_id').set_index('uniprot_id'))
frac = pd.concat(frac, axis=1)
frac['frac_cyto'] = frac[['frac_cyto_hek', 'frac_cyto_hela']].mean(axis=1)
frac['frac_nuc'] = frac[['frac_nuc_hek', 'frac_nuc_hela']].mean(axis=1)
ctx = ctx.join(frac[['frac_cyto', 'frac_nuc']], on='uniprot_id')
print(f"Quota citosolica/nucleare misurata: {ctx['frac_cyto'].notna().sum():,} proteine (linee HEK293T/HeLa: non sono neuroni → limite dichiarato)")

Quota citosolica/nucleare misurata: 9,438 proteine (linee HEK293T/HeLa: non sono neuroni → limite dichiarato)


## Coorte: umane, cervello, senza sangue + validazione
Funnel dei filtri stampato a ogni passo. Il test HPA indipendente è marcato `role = test` e **non compare mai** nelle osservazioni di addestramento (controllo con `assert`).

In [7]:
# ---------- COORTE: proteine umane, espresse nel cervello, senza sangue ------------------------------------------------------
funnel = [('UniProt umane reviewed', len(ctx))]
m_len = ctx['uniprot_id'].map(up.set_index('uniprot_id')['sequence'].str.len()).between(41, MAX_LEN)
steps = [('lunghezza 41–%d aa' % MAX_LEN, m_len),
         ('senza sangue', (ctx['is_blood'] == 0) | (not ESCLUDI_SANGUE)),
         ('espresse nel cervello (HPA tessuto, HPA tipo cellulare o UniProt)', (ctx['is_brain'] + ctx['is_neuronal'] + ctx['is_oligo']) > 0)]
keep = pd.Series(True, index=ctx.index)
for name, m in steps:
    keep &= m
    funnel.append((name, int(keep.sum())))
cohort = ctx[keep].copy()

obs_ids = set(obs['uniprot_id'])
test_ids = set(hpa_test['uniprot_id'].dropna()) & set(cohort['uniprot_id'])
cohort['role'] = np.where(cohort['uniprot_id'].isin(test_ids), 'test', 'train')         # il test HPA non entra MAI nell'addestramento
cohort['has_label'] = cohort['uniprot_id'].isin(obs_ids) & (cohort['role'] == 'train')
funnel += [('di cui con almeno un\'osservazione (addestrabili)', int(cohort['has_label'].sum())),
           ('di cui test indipendente HPA', len(test_ids))]
print(pd.DataFrame(funnel, columns=['passo', 'proteine']).to_string(index=False))

# etichette del test HPA (DeepLoc 2.0): solo le classi che l'HPA misura al microscopio
ht = hpa_test.dropna(subset=['uniprot_id']).drop_duplicates('uniprot_id').set_index('uniprot_id')
for c in ORGANELLES:
    cohort['test_' + c] = cohort['uniprot_id'].map(ht[c]) if (c in ht.columns and c != 'Extracellular') else np.nan
obs = obs[obs['uniprot_id'].isin(set(cohort.loc[cohort['role'] == 'train', 'uniprot_id']))].copy()
cohort = cohort.merge(up[['uniprot_id', 'sequence', 'function', 'disease', 'subunit', 'ptm', 'tissue', 'catalytic', 'pathway', 'actreg',
                          'family', 'domain', 'repeat', 'zinc', 'coiled', 'region', 'motif', 'keywords', 'ec']], on='uniprot_id', how='left')
cohort = cohort.reset_index(drop=True)
cohort['pid'] = np.arange(len(cohort))
obs['pid'] = obs['uniprot_id'].map(cohort.set_index('uniprot_id')['pid'])

# ---------- /validate-data: controlli che fermano il notebook se qualcosa non torna --------------------------------------------
assert cohort['uniprot_id'].is_unique, 'UniProt duplicati nella coorte'
assert cohort['sequence'].str.fullmatch(r'[A-Z]+').all(), 'sequenze con caratteri non validi'
assert not set(obs['uniprot_id']) & test_ids, 'LEAK: proteine del test HPA presenti nelle osservazioni di addestramento'
assert obs['pid'].notna().all() and obs['obs'].isin([0.0, 1.0]).all()
assert (not ESCLUDI_SANGUE) or (cohort['is_blood'] == 0).all(), 'proteine del sangue rimaste nella coorte'
print('Validazione OK · proteine:', len(cohort), '· osservazioni:', len(obs))

                                                            passo  proteine
                                           UniProt umane reviewed     20430
                                            lunghezza 41–12000 aa     20358
                                                     senza sangue     17649
espresse nel cervello (HPA tessuto, HPA tipo cellulare o UniProt)     13298
                 di cui con almeno un'osservazione (addestrabili)     13043
                                     di cui test indipendente HPA       255
Validazione OK · proteine: 13298 · osservazioni: 439719


## AlphaFold: feature strutturali
L'API restituisce il PDB; il fattore B è il pLDDT. Le feature sono calcolate dai Cα (neighbor-count a 10 Å come proxy di sepoltura, geometria Cα(i)–Cα(i+3) per le eliche). Funzione testata su una struttura sintetica prima dell'uso.

In [ ]:
# ---------- AlphaFold DB: struttura 3D e confidenza (pLDDT) per ogni proteina della coorte ---------------------------------------
# API: https://alphafold.ebi.ac.uk/api/prediction/<UniProt>  →  JSON con l'URL del PDB; il fattore B del PDB è il pLDDT per residuo.
# Dal PDB si ricavano feature che il language model non vede: accessibilità di N- e C-terminale (un motivo C-terminale
# come PTS1/PDZ/KDEL funziona solo se è esposto), regioni disordinate (condensati, granuli di RNA), eliche TM verificate sulla
# geometria, e la "toppa" di lisine esposte che il lisosoma riconosce (mannosio-6-fosfato) sulla proteina ripiegata.
AA3 = {'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C', 'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
       'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'}
KD_AF = dict(zip('ACDEFGHIKLMNPQRSTVWY', [1.8, 2.5, -3.5, -3.5, 2.8, -0.4, -3.2, 4.5, -3.9, 3.8, 1.9, -3.5, -1.6, -3.5, -4.5, -0.8, -0.7, 4.2, -0.9, -1.3]))
AF_COLS = ['af_mean_plddt', 'af_frac_vlow', 'af_frac_low', 'af_frac_conf', 'af_frac_vhigh', 'af_plddt_N', 'af_plddt_M', 'af_plddt_C',
           'af_plddt_last5', 'af_contact_density', 'af_expo_N10', 'af_expo_C5', 'af_rg_norm', 'af_helix_frac', 'af_strand_frac',
           'af_longest_disorder', 'af_n_idr30', 'af_n_long_helix', 'af_n_tm_like_helix', 'af_k_patch', 'af_basic_patch',
           'af_exposed_hydrophobic', 'af_partial']

def parse_pdb_ca(text):
    xyz, pl, aa = [], [], []
    for ln in text.splitlines():
        if ln.startswith('ATOM') and ln[12:16] == ' CA ':
            xyz.append((float(ln[30:38]), float(ln[38:46]), float(ln[46:54])))
            pl.append(float(ln[60:66]))
            aa.append(AA3.get(ln[17:20], 'X'))
    return np.array(xyz), np.array(pl), ''.join(aa)

def runs(mask):
    """Lunghezze e inizi delle serie consecutive di True."""
    m = np.concatenate([[0], mask.astype(int), [0]])
    d = np.diff(m)
    st, en = np.flatnonzero(d == 1), np.flatnonzero(d == -1)
    return st, en - st

def af_features(xyz, pl, aa):
    n = len(pl)
    f = {'af_mean_plddt': pl.mean(), 'af_frac_vlow': (pl < 50).mean(), 'af_frac_low': ((pl >= 50) & (pl < 70)).mean(),
         'af_frac_conf': ((pl >= 70) & (pl < 90)).mean(), 'af_frac_vhigh': (pl >= 90).mean(),
         'af_plddt_N': pl[:30].mean(), 'af_plddt_C': pl[-30:].mean(), 'af_plddt_last5': pl[-5:].mean()}
    m0 = max(0, n // 2 - 15)
    f['af_plddt_M'] = pl[m0:m0 + 30].mean()
    nb = cKDTree(xyz).query_ball_point(xyz, 10.0, return_length=True) - 1          # Cα entro 10 Å: proxy di sepoltura
    expo = np.clip((30 - nb) / 18.0, 0, 1)                                         # 1 = esposto, 0 = sepolto (proxy, non DSSP)
    f['af_contact_density'] = nb.mean()
    f['af_expo_N10'], f['af_expo_C5'] = expo[:10].mean(), expo[-5:].mean()
    f['af_rg_norm'] = np.sqrt(((xyz - xyz.mean(0)) ** 2).sum(1).mean()) / n ** 0.38     # ~ costante per proteine globulari
    d3 = np.linalg.norm(xyz[3:] - xyz[:-3], axis=1) if n > 4 else np.array([])
    d2 = np.linalg.norm(xyz[2:] - xyz[:-2], axis=1) if n > 4 else np.array([])
    helix = np.zeros(n, bool)
    helix[:len(d3)] = (d3 > 4.8) & (d3 < 5.9)                                      # Cα(i)–Cα(i+3) ≈ 5,0–5,5 Å nelle α-eliche
    helix &= pl > 70
    strand = np.zeros(n, bool)
    strand[:len(d2)] = d2 > 6.2
    strand &= (pl > 70) & ~helix
    f['af_helix_frac'], f['af_strand_frac'] = helix.mean(), strand.mean()
    st, ln = runs(pl < 50)
    f['af_longest_disorder'] = (ln.max() / n) if len(ln) else 0.0
    f['af_n_idr30'] = int((ln >= 30).sum())
    hs, hl = runs(helix)
    long_h = [(s, l + 3) for s, l in zip(hs, hl) if l + 3 >= 17]                   # eliche lunghe ≥17 residui
    f['af_n_long_helix'] = len(long_h)
    f['af_n_tm_like_helix'] = sum(np.mean([KD_AF.get(a, 0) for a in aa[s:s + l]]) >= 1.2 for s, l in long_h)   # lunghe E idrofobiche
    folded = pl > 70
    for letters, name in [('K', 'af_k_patch'), ('KR', 'af_basic_patch')]:
        idx = np.flatnonzero(np.array([a in letters for a in aa]) & folded & (expo > 0.5))
        f[name] = int(cKDTree(xyz[idx]).query_ball_point(xyz[idx], 15.0, return_length=True).max()) if len(idx) > 1 else 0
    surf = folded & (expo > 0.5)
    f['af_exposed_hydrophobic'] = np.mean([a in 'AILMFVWC' for a in np.array(list(aa))[surf]]) if surf.any() else 0.0
    return f

def af_fetch(acc):
    """Scarica (con cache) JSON e PDB di una proteina; restituisce le feature o None se AlphaFold non la ha."""
    jf, pf = AFDIR / f'{acc}.json', AFDIR / f'{acc}.pdb'
    if not pf.exists():
        if not jf.exists():
            r = session.get(f'https://alphafold.ebi.ac.uk/api/prediction/{acc}', timeout=60)
            if r.status_code == 404:
                jf.write_text('[]')
            else:
                r.raise_for_status()
                jf.write_text(r.text)
        meta = json.loads(jf.read_text())
        if not meta:
            return None
        url = meta[0].get('pdbUrl') or f'https://alphafold.ebi.ac.uk/files/AF-{acc}-F1-model_v4.pdb'
        r = session.get(url, timeout=120)
        r.raise_for_status()
        pf.write_text(r.text)
    xyz, pl, aa = parse_pdb_ca(pf.read_text())
    if len(pl) < 10:
        return None
    return af_features(xyz, pl, aa)

af = pd.DataFrame(columns=['uniprot_id'] + AF_COLS)
if USE_ALPHAFOLD:
    ids = cohort['uniprot_id'].tolist()[:AF_LIMIT]
    def job(acc):
        try:
            return acc, af_fetch(acc)
        except Exception as e:                                   # rete o file corrotto: la proteina resta senza struttura
            return acc, None
    t0, rows = time.time(), []
    with ThreadPoolExecutor(AF_WORKERS) as ex:
        for i, (acc, f) in enumerate(ex.map(job, ids), 1):
            if f is not None:
                rows.append({'uniprot_id': acc, **f})
            if i % 1000 == 0:
                print(f'AlphaFold {i:,}/{len(ids):,} ({time.time() - t0:.0f}s)')
    af = pd.DataFrame(rows)
    seq_len = cohort.set_index('uniprot_id')['sequence'].str.len()
    pdb_len = af['uniprot_id'].map(lambda a: len(parse_pdb_ca((AFDIR / f'{a}.pdb').read_text())[1]))
    af['af_partial'] = (pdb_len / af['uniprot_id'].map(seq_len) < 0.9).astype(float)      # struttura che non copre la proteina (frammenti)
    print(f"Strutture AlphaFold: {len(af):,}/{len(ids):,} proteine ({len(af) / max(len(ids), 1):.1%})")
cohort = cohort.merge(af, on='uniprot_id', how='left')

AlphaFold 1,000/13,298 (1205s)


## `card_protein`: il profilo testuale
Sei sezioni: *identità e funzione · domini e motivi · malattie neurologiche · regolazione e partner · espressione cerebrale · profilo esperto (regole + AlphaFold)*.

**Due versioni per un motivo scientifico.** UniProt scrive spesso la localizzazione dentro il testo della funzione. `card_protein` (input del modello) elimina frasi, parole chiave, nomi e motivi che nominano un compartimento o un meccanismo di sorting; `card_full` conserva tutto. Nel notebook 02 la differenza tra le due misura quanta parte della prestazione sarebbe leakage. Non è un filtro perfetto: la *famiglia* ("nucleoporina") e la funzione ("fattore di trascrizione") restano, perché sono biologia legittima; il notebook 02 valuta anche le proteine con card povera per stimare la prestazione su proteine poco annotate.

In [ ]:
# ---------- card_protein: il profilo testuale di ogni proteina --------------------------------------------------------------------
# La card riassume ciò che un esperto di biologia molecolare e neurologia vorrebbe sapere: identità e famiglia, funzione molecolare,
# domini, regolazione e PTM, partner/complessi, malattie neurologiche, espressione cerebrale, profilo strutturale (AlphaFold) e una
# valutazione "da esperto" a regole (classe funzionale, rilevanza neuronale).
#
# ⚠ RISCHIO DI LEAKAGE: UniProt scrive spesso "localizzata nella densità postsinaptica" proprio nel testo della funzione.
#   Dare questo testo al modello vorrebbe dire dargli la risposta. Per questo esistono DUE versioni:
#   · card_protein  → ripulita: via le frasi, le parole chiave, i nomi e i motivi che dichiarano un compartimento (è l'INPUT del modello)
#   · card_full     → completa, per la lettura umana e come "controllo leakage" nel notebook 02 (la differenza di prestazioni misura la fuga)
LOC_TERMS = (r'subcellular|locali[sz]|located|translocat|targeted? to|targeting|nucle(?:us|ar|oplasm|olus|olar)|cytoplasm|cytosol|cytoskelet|'
             r'mitochondri|endoplasmic|\bER\b|golgi|lysosom|endosom|vacuol|peroxisom|secreted|extracellular|plasma membrane|cell membrane|'
             r'cell surface|synap|dendrit|axon|myelin|\bspines?\b|vesicle|organelle|cilium|ciliary|cell junction|cell projection|neurite|'
             r'growth cone|soma\b|perikaryon|exosome|signal peptide|transit peptide|signal sequence|transmembrane|single-pass|multi-pass|'
             r'localization signal|export signal|microbody|retention signal|prevents secretion|sarcolemma|cell periphery|membrane-bound|'
             r'glycosyl|n-linked|o-linked|disulfide|gpi|palmitoyl|myristoyl|prenyl|farnesyl|lipidat|\bmembrane\b|lumen|luminal')
LOC_RE = re.compile(LOC_TERMS, re.I)

def clean(text):
    """Toglie codici di evidenza, riferimenti PubMed, prefissi UniProt e spazi doppi."""
    if not isinstance(text, str) or not text.strip():
        return ''
    t = re.sub(r'\{ECO:[^}]*\}', '', text)
    t = re.sub(r'\((?:PubMed|Ref\.)[^)]*\)', '', t)
    t = re.sub(r'\b(?:FUNCTION|DISEASE|SUBUNIT|PTM|TISSUE SPECIFICITY|CATALYTIC ACTIVITY|PATHWAY|ACTIVITY REGULATION|DOMAIN)\s*:', ' ', t)
    t = re.sub(r'\[MIM:\d+\]', ' ', t)
    t = re.sub(r'\bReaction=', ' ', t)
    t = re.sub(r'\s+([:.,;])', r'\1', t)
    return re.sub(r'\s+', ' ', t).strip(' .;')

def scrub(text):
    """Elimina le frasi che nominano un compartimento cellulare."""
    sents = re.split(r'(?<=[.;])\s+', clean(text))
    return ' '.join(s for s in sents if s and not LOC_RE.search(s)).strip(' .;')

NAME_DROP = (r'\b(?:mitochondrial|nuclear|synaptic|lysosomal|peroxisomal|microsomal|membrane|secreted|extracellular|cytoplasmic|vesicular|'
             r'vesicle|presynaptic|postsynaptic|neuronal|Golgi|endoplasmic reticulum|plasma|cell surface)\b')

def scrub_name(name):
    n = re.sub(r'\([^)]*\)', ' ', name if isinstance(name, str) else '')
    n = re.sub(NAME_DROP, ' ', n, flags=re.I)
    n = ', '.join(p.strip() for p in n.split(',') if p.strip())
    return re.sub(r'\s+', ' ', n).strip(' ,')

def feature_names(text):
    """Nomi dei domini/regioni/motivi da un campo FT di UniProt (solo i nomi, senza coordinate)."""
    if not isinstance(text, str):
        return []
    names = re.findall(r'/note="([^"]*)"', text)
    return list(dict.fromkeys(re.sub(r'\s*\d+$', '', n).strip() for n in names))

# classi funzionali a regole (conoscenza di dominio: cosa fa la proteina, non dove sta)
FUNC_RULES = [('chinasi', r'\bkinase\b'), ('fosfatasi', r'phosphatase'), ('proteasi/peptidasi', r'protease|peptidase|proteinase|caspase'),
              ('GTPasi / regolatore GTPasico', r'\bgtpase\b|\bGEF\b|\bGAP\b|guanine nucleotide exchange|rab |ras |rho |arf '),
              ('ubiquitina ligasi / deubiquitinasi', r'ubiquitin|e3 ligase|deubiquitin|sumo'), ('canale ionico', r'ion channel|\bchannel\b|voltage-gated|ligand-gated'),
              ('trasportatore', r'transporter|symporter|antiporter|solute carrier|\bpump\b|atpase'), ('recettore', r'\breceptor\b'),
              ('GPCR', r'g-protein coupled|g protein-coupled|gpcr'), ('fattore di trascrizione / regolatore della cromatina', r'transcription factor|transcription|chromatin|histone|dna-binding|dna binding|helicase'),
              ('legame all\'RNA / splicing', r'rna-binding|rna binding|splicing|mrna|translation|ribosom'), ('chaperone / proteostasi', r'chaperone|heat shock|proteasom|autophag'),
              ('proteina di impalcatura / adattatore', r'scaffold|adaptor|adapter|anchor|pdz|sh3|ankyrin'), ('citoscheletro / motore', r'tubulin|actin|microtubule|kinesin|dynein|myosin|tau '),
              ('fusione vescicolare / SNARE', r'snare|synaptotagmin|exocytosis|endocytosis|vesicle fusion|neurotransmitter release|clathrin|dynamin'),
              ('adesione cellulare / matrice', r'adhesion|cadherin|integrin|laminin|collagen|neurexin|neuroligin'),
              ('ormone / neuropeptide / fattore secreto', r'hormone|neuropeptide|neurotrophic|growth factor|cytokine|chemokine'),
              ('enzima metabolico', r'dehydrogenase|oxidase|reductase|synthase|synthetase|transferase|hydrolase|lyase|isomerase|ligase'),
              ('metabolismo lipidico', r'lipid|sphingo|cholesterol|fatty acid|phospholip'), ('mitocondriale: fosforilazione ossidativa', r'respiratory|oxidative phosphorylation|complex i\b|atp synthase|cytochrome')]
NEURO_DISEASES = [('Alzheimer / demenza', r'alzheimer|dementia|tauopath'), ('Parkinson / parkinsonismo', r'parkinson|lewy'),
                  ('SLA / demenza frontotemporale', r'amyotrophic|frontotemporal'), ('Huntington / malattie da poliglutammine', r'huntington|polyglutamine'),
                  ('epilessia / encefalopatia epilettica', r'epilep|seizure|epileptic encephalopathy'), ('disabilità intellettiva / neurosviluppo', r'intellectual disability|neurodevelopmental|developmental delay|autism'),
                  ('atassia', r'ataxia'), ('neuropatia periferica (CMT)', r'charcot|neuropathy|hereditary motor'), ('leucodistrofia / mielina', r'leukodystroph|demyelinat|myelin'),
                  ('paraplegia spastica', r'spastic paraplegia'), ('malformazioni corticali / microcefalia', r'microcephal|lissencephal|cortical malformation|heterotopia|polymicrogyria'),
                  ('malattie lisosomiali / ceroidolipofuscinosi', r'lysosomal storage|ceroid|neuronal ceroid|gangliosidosis|mucopolysaccharid'),
                  ('malattie perossisomiali', r'zellweger|peroxisom|adrenoleukodystrophy'), ('malattia mitocondriale / Leigh', r'leigh|mitochondrial (?:complex|disease)|mitochondrial dna'),
                  ('canalopatia', r'channelopathy|episodic ataxia|periodic paralysis'), ('psichiatria (schizofrenia, bipolare)', r'schizophren|bipolar|psychiatric'),
                  ('distonia / disturbi del movimento', r'dystonia|chorea|tremor|myoclonus')]

def expert_profile(r):
    """Valutazione da esperto: cosa fa la proteina, quanto è rilevante per il cervello, com'è fatta (AlphaFold)."""
    blob = ' '.join(str(r.get(k, '')) for k in ['protein_name', 'function', 'family', 'keywords', 'catalytic', 'ec', 'domain']).lower()
    funcs = [n for n, p in FUNC_RULES if re.search(p, blob)]
    out = []
    out.append('Classe funzionale (regole): ' + (', '.join(funcs[:4]) if funcs else 'non classificata') + '.')
    flags = []
    if r.get('is_neuronal', 0) == 1:
        flags.append('espressa nei neuroni')
    if r.get('is_oligo', 0) == 1:
        flags.append('espressa negli oligodendrociti')
    if r.get('aux_patologia_cerebrale', 0) == 1:
        flags.append('associata a patologia cerebrale')
    if r.get('aux_aggregati', 0) == 1:
        flags.append('legata ad aggregati proteici/amiloide')
    out.append('Rilevanza neurologica: ' + ('; '.join(flags) if flags else 'nessun segnale esplicito') + '.')
    ln = len(r['sequence'])
    out.append(f"Dimensione: {ln} aa ({'molto grande, tipica di scaffold e motori' if ln > 1500 else 'grande' if ln > 600 else 'media' if ln > 250 else 'piccola'}).")
    if r.get('aux_multimero', 0) == 1:
        out.append('Forma oligomeri o complessi.')
    if pd.notna(r.get('af_mean_plddt')):
        dis = r['af_frac_vlow'] + r['af_frac_low']
        out.append(f"Struttura AlphaFold: pLDDT medio {r['af_mean_plddt']:.0f}; {dis:.0%} dei residui a bassa confidenza (disordine)"
                   f"{'; proteina in gran parte intrinsecamente disordinata (candidata a condensati/granuli)' if dis > 0.5 else ''}; "
                   f"{int(r['af_n_tm_like_helix'])} eliche lunghe idrofobiche (compatibili con segmenti transmembrana); "
                   f"terminale C {'esposto' if r['af_expo_C5'] > 0.6 else 'parzialmente sepolto'}.")
    return ' '.join(out)

def build_card(r, full=False):
    f = (lambda t: clean(t)) if full else scrub
    name = clean(r['protein_name']) if full else scrub_name(r['protein_name'])
    fam = f(r['family'])
    sec = {}
    sec['identita_funzione'] = ' '.join(x for x in [
        f"Proteina {name}" + (f" (gene {r['gene']})" if r['gene'] else '') + '.',
        f"Famiglia: {fam}." if fam else '', f"Funzione: {f(r['function'])}." if f(r['function']) else '',
        f"Attività catalitica: {f(r['catalytic'])}." if f(r['catalytic']) else '', f"Via metabolica: {f(r['pathway'])}." if f(r['pathway']) else ''] if x)
    doms = [d for d in feature_names(r['domain']) + feature_names(r['repeat']) + feature_names(r['zinc']) + feature_names(r['coiled']) + feature_names(r['region']) + feature_names(r['motif'])
            if full or not LOC_RE.search(d)]
    sec['domini_motivi'] = ('Domini, ripetizioni e regioni: ' + '; '.join(doms[:25]) + '.') if doms else ''
    dtxt = f(r['disease'])
    tags = [n for n, p in NEURO_DISEASES if re.search(p, str(r['disease']), re.I)]
    sec['malattie'] = ' '.join(x for x in [('Malattie neurologiche associate: ' + ', '.join(tags) + '.') if tags else '',
                                          f"Coinvolgimento in malattia: {dtxt}." if dtxt else ''] if x)
    sec['regolazione_partner'] = ' '.join(x for x in [
        f"Regolazione: {f(r['actreg'])}." if f(r['actreg']) else '', f"Modifiche post-traduzionali: {f(r['ptm'])}." if f(r['ptm']) else '',
        f"Struttura quaternaria e partner: {f(r['subunit'])}." if f(r['subunit']) else ''] if x)
    kw = [k.strip() for k in str(r['keywords']).split(';') if k.strip() and k.strip().lower() != 'nan' and (full or not LOC_RE.search(k))]
    expr = []
    if r['brain_regions']:
        expr.append(f"Espressione RNA nel cervello (HPA): {r['brain_regions']}.")
    if r['brain_celltypes']:
        expr.append(f"Tipi cellulari cerebrali (HPA single-cell): {r['brain_celltypes']}.")
    if f(r['tissue']):
        expr.append(f"Specificità tissutale: {f(r['tissue'])}.")
    if kw:
        expr.append('Parole chiave UniProt: ' + ', '.join(kw[:20]) + '.')
    sec['espressione'] = ' '.join(expr)
    sec['profilo_esperto'] = expert_profile(r)
    return sec

SECTIONS = ['identita_funzione', 'domini_motivi', 'malattie', 'regolazione_partner', 'espressione', 'profilo_esperto']
rows = cohort.fillna({c: '' for c in ['protein_name', 'function', 'disease', 'subunit', 'ptm', 'tissue', 'catalytic', 'pathway', 'actreg', 'family',
                                       'domain', 'repeat', 'zinc', 'coiled', 'region', 'motif', 'keywords', 'ec', 'gene']}).to_dict('records')
safe = [build_card(r) for r in rows]
full_ = [build_card(r, full=True) for r in rows]
for s in SECTIONS:
    cohort['card_' + s] = [c[s] for c in safe]
cohort['card_protein'] = ['\n'.join(f"[{s}] {c[s]}" for s in SECTIONS if c[s]) for c in safe]
cohort['card_full'] = ['\n'.join(f"[{s}] {c[s]}" for s in SECTIONS if c[s]) for c in full_]
cohort['card_n_words'] = cohort['card_protein'].str.split().str.len()
print(cohort['card_n_words'].describe().round(0).to_string())
leak = cohort['card_protein'].str.contains(LOC_RE)
print(f'Controllo leakage: card_protein contiene termini di compartimento in {leak.sum()} proteine su {len(cohort)}')
print('\n--- esempio ---\n', cohort.sort_values('card_n_words').iloc[len(cohort) // 2][['gene', 'card_protein']].to_string())

## /explore-data e salvataggio
Bilanciamento delle classi, distribuzione dell'espressione e scrittura dei file per il notebook 02.

In [ ]:
import matplotlib.pyplot as plt

# ---------- /explore-data: quante etichette per classe e per fonte (le classi sono equilibrate abbastanza?) ----------------------
tab = obs.pivot_table(index='fonte', columns='classe', values='obs', aggfunc='sum').reindex(columns=ALL).fillna(0).astype(int)
print('Positivi per fonte e classe:\n', tab.T.to_string())
strong_src = ['deeploc', 'uniprot_forte', 'hpa_alta', 'mitocarta', 'syngo', 'goa_exp']
pos_any = obs[obs['fonte'].isin(strong_src) & (obs['obs'] == 1)].groupby('classe')['pid'].nunique().reindex(ALL).fillna(0).astype(int)
neu_ok = cohort.set_index('pid')['is_neuronal']
print('\nProteine con almeno un positivo da una fonte forte, per classe (la coorte ha', len(cohort), 'proteine):')
print(pos_any.to_string())
low = pos_any[pos_any < 150].index.tolist()
print('\nClassi con < 150 positivi (stime più incerte, il notebook 02 le segnala con intervalli di confidenza ampi):', low or 'nessuna')

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
pos_any.plot.barh(ax=ax[0], color=['#378ADD'] * 9 + ['#1D9E75'] * 5)
ax[0].set_title('Positivi per classe (fonti forti)'); ax[0].invert_yaxis()
cohort[['brain_ntpm', 'neuron_ncpm']].apply(lambda s: np.log10(s.fillna(0) + 1)).plot.hist(bins=40, alpha=.6, ax=ax[1])
ax[1].set_title('Espressione cerebrale della coorte (log10)')
plt.tight_layout(); plt.savefig(DATA / 'eda_classi.png', dpi=130); plt.show()

# ---------- salvataggio ---------------------------------------------------------------------------------------------------------------
keep_cols = (['pid', 'uniprot_id', 'gene', 'protein_name', 'sequence', 'role', 'has_label', 'is_brain', 'is_neuronal', 'is_oligo', 'is_blood',
              'brain_ntpm', 'neuron_ncpm', 'glia_ncpm', 'frac_cyto', 'frac_nuc', 'family', 'card_protein', 'card_full', 'card_n_words']
             + ['card_' + s for s in SECTIONS] + [c for c in cohort.columns if c.startswith(('aux_', 'ctx_', 'test_', 'af_'))])
cohort[keep_cols].to_parquet(DATA / 'proteine.parquet', index=False)
obs[['pid', 'uniprot_id', 'fonte', 'classe', 'obs']].to_parquet(DATA / 'osservazioni.parquet', index=False)
json.dump({'POSITIVE_ONLY': POSITIVE_ONLY, 'ALL': ALL, 'ORGANELLES': ORGANELLES, 'NEURO': NEURO}, open(DATA / 'config_dati.json', 'w'))
print(f"Salvati: data/proteine.parquet ({len(cohort):,} proteine), data/osservazioni.parquet ({len(obs):,} osservazioni)")